In [2]:
%%capture --no-stderr
%pip install --quiet -U langchain_openai langchain_core langchain_community tavily-python langchain-tavily rapidfuzz langgraph

In [91]:
import os, getpass
from dotenv import load_dotenv
from tavily import TavilyClient
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch, TavilyExtract

# Load environment variables from .env file
load_dotenv(dotenv_path="etl/.env")

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")
_set_env("LANGCHAIN_API_KEY")
_set_env("TAVILY_API_KEY")

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "runner_agent"

LANGSMITH_ENDPOINT="https://api.smith.langchain.com"
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

# Tavily tools
search = TavilySearch(max_results=4, search_depth="basic", include_raw_content="text")
extract = TavilyExtract(extract_depth="basic")
tools = [search, extract]

reasoning_llm = ChatOpenAI(model="gpt-4.1", api_key=os.getenv("OPENAI_API_KEY"))
llm = ChatOpenAI(model="gpt-4.1-nano-2025-04-14", api_key=os.getenv("OPENAI_API_KEY"))

In [92]:
from typing import Any, Dict, Annotated
from typing_extensions import TypedDict, NotRequired
from langgraph.graph import MessagesState
from langchain_core.messages import HumanMessage, SystemMessage


class RunnerProfileState(TypedDict):
    # Inputs
    runner_name: str
    college_name: str
    search_results: NotRequired[list[str]]
    page_snippets: NotRequired[list[dict]]      # {url, snippet}
    selected_profile_url: NotRequired[str]
    full_profile_text: NotRequired[str]
    hometown: NotRequired[str]
    high_school: NotRequired[str]
    swim_background: NotRequired[str]

    swim_search_results: NotRequired[list[dict]]  # raw hits: {url, title, content, score}
    swim_candidates:   NotRequired[list[str]]     # URLs that passed filter
    swim_profile_url:  NotRequired[str]           # top candidate (or null)
    swim_profile_text: NotRequired[str]           # full extracted page text

    has_swim_background:  NotRequired[bool]
    rationale:            NotRequired[str]

    # Errors
    error: NotRequired[str]

In [103]:
import json, re, httpx, asyncio
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain import LLMChain

# 1. search_pages
def search_pages(state: RunnerProfileState, config: Any) -> Dict[str, Any]:
    query = f"{state['runner_name']} {state['college_name']} profile"
    res = search.invoke(query)
    urls = [hit["url"] for hit in res.get("results", [])]
    # Add raw_content to each snippet dict
    snippets = [
        {
            "url": hit["url"],
            "snippet": hit.get("content", "")[:2000],
            "raw_content": hit.get("raw_content", "")
        }
        for hit in res.get("results", [])
    ]
    return {
        "search_results": urls,
        "page_snippets": snippets
    }

# 2. select_profile_page
select_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are looking for the official college roster/profile page. "
     "Given these URL+snippet pairs for {runner_name} at {college_name}, "
     "pick the one that is the athlete’s roster page on the college site. "
     "Return just the URL or null if none."),
    ("human", "Pages:\n{pages}")
])

def select_profile_page(state: RunnerProfileState, config: Any) -> Dict[str, Any]:
    pages = state.get("page_snippets", [])
    if not pages:
        return {"error": "No pages to choose from."}
    blurb = "\n".join(
        f"[{i}] {p['url']}\n{p['snippet']}\n"
        for i, p in enumerate(pages, start=1)
    )
    prompt_input = select_prompt.invoke({
        "pages": blurb,
        "runner_name": state["runner_name"],
        "college_name": state["college_name"]
    })
    response = llm.invoke(prompt_input)
    chosen = response.content.strip()
    selected_url = chosen if chosen != "null" else None
    # Find raw_content for the selected URL
    '''full_profile_text = None
    if selected_url:
        for p in pages:
            if p["url"] == selected_url:
                full_profile_text = p.get("raw_content", "")
                break'''
    return {
        "selected_profile_url": selected_url,
        #"full_profile_text": full_profile_text
    }

def fetch_full_profile(state: RunnerProfileState, config: Any) -> Dict[str, Any]:
    url = state.get("selected_profile_url")
    if not url:
        return {"error": "No profile URL selected."}
    res = extract.invoke({"urls": [url]})
    if isinstance(res, str):
        res = json.loads(res)

    first = res["results"][0]
    full = first.get("raw_content") or first.get("content") or ""

    return {"full_profile_text": full}
    
    

parse_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an expert data extractor. "
     "Given the full text of an athlete’s official roster page, "
     "extract hometown, high school, and any swim experience or interest. "
     "Highschool should be a short name, and not a list of accomplishments from high school. "
     "Return JSON with keys hometown, high_school, swim_background (short sentence or null). "
     "For swim_background, return 'yes' if there is any mention of swimming experience or interest, otherwise return 'no'."),
    ("human", "{profile_text}")
])

def safe_json_parse(text: str) -> dict:
    """
    Safely parse a JSON string, returning a dict.
    If parsing fails, returns {'error': 'Could not parse JSON'}
    """
    try:
        return json.loads(text)
    except Exception as e:
        return {"error": f"Could not parse JSON: {e}"}

def parse_metadata(state: RunnerProfileState, config: Any) -> Dict[str, Any]:
    text = state.get("full_profile_text", "")
    if not text:
        return {"error": "No full page text to parse."}
    prompt_input = parse_prompt.invoke({"profile_text": text})
    response = llm.invoke(prompt_input)
    data = safe_json_parse(response.content)
    if "error" in data:
        return data
    return {
        "hometown": data.get("hometown"),
        "high_school": data.get("high_school"),
        "swim_background": data.get("swim_background")
    }

In [104]:
def search_swimcloud(state: RunnerProfileState, config: Any) -> dict:
    # build the query from demographic state
    name     = state["runner_name"]
    hometown = state.get("hometown", "")
    query    = f"{name} {hometown} swimcloud"

    # run TavilySearch
    res = search.invoke(query)
    hits = [
        {
            "url":     hit["url"],
            "title":   hit.get("title", ""),
            "content": hit.get("content", ""),
            "score":   hit.get("score", 0.0),
            "raw content": hit.get("raw_content", "")
        }
        for hit in res.get("results", [])
    ]

    candidates = [
        h["url"] for h in hits
        if h["score"] >= 0. and "swimcloud.com" in h["url"]
    ]
    top = candidates[0] if candidates else None

    swim_profile_text = None
    if top:
        for hit in hits:
            if hit["url"] == top:
                swim_profile_text = hit.get("raw content", "")
                break
    

    return {"swim_search_results": hits,
            "swim_candidates": candidates,
            "swim_profile_url": top,
            "swim_profile_text": swim_profile_text}  # will be filled later


In [98]:
reasoning_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an expert triathlon talent scout. "
     "Your task is to determine whether a collegiate runner may hava a background or interest in swimming."
     "You will be given the runner’s college profile page text and a potential matching SwimCloud profile page text,"
     "The Swimcloud profile page may match the runner's name, but it may not be the same person. "
     "Use the context at hand to determine definitively whether the runner has a swimming background or serious interest. "
     "Return a JSON object with keys:\n"
     "  has_swim_background (boolean),\n"
     "  rationale (string) — one or two sentences summarizing the evidence."),
    ("human",
     "Runner Profile Page:\n```\n{profile_text}\n```\n\n"
     "SwimCloud Profile Page:\n```\n{swim_text}\n```")
])

def reason_about_swim_background(state: RunnerProfileState, config: Any) -> dict:
    profile = state.get("full_profile_text", "")
    swim = state.get("swim_profile_text", "")
    swim_interest = state.get("swim_background", "")
    if not profile or not swim:
        return {
            "has_swim_background": False,
            "rationale": "No profile or swim page text available."
        }
    if swim_interest == "yes":
        # If we already have swim interest, return it directly
        return {
            "has_swim_background": True,
            "rationale": "Runner profile had mention of swimming."
        }

    # Modern prompt invocation pattern
    prompt_input = reasoning_prompt.invoke({
        "profile_text": profile[:15000],
        "swim_text": swim[:15000]
    })
    response = reasoning_llm.invoke(prompt_input)
    llm_output = response.content.strip()

    # Parse the JSON output
    try:
        result = json.loads(llm_output)
    except json.JSONDecodeError:
        # Fallback: assume false if the model garbled
        return {
            "has_swim_background": False,
            "rationale": "Could not parse LLM response."
        }

    return {
        "has_swim_background": result.get("has_swim_background", False),
        "rationale": result.get("rationale", "")
    }

In [105]:
from langgraph.graph import StateGraph
from IPython.display import display, Image

builder = StateGraph(RunnerProfileState)
builder.add_node("search",    search_pages)
builder.add_node("select",    select_profile_page)
builder.add_node("fetch",     fetch_full_profile)
builder.add_node("parse",     parse_metadata)
builder.add_node("search_swimcloud", search_swimcloud)
builder.add_node("reasoning", reason_about_swim_background)

builder.add_edge("search", "select")
builder.add_edge("select", "fetch")
builder.add_edge("fetch", "parse")
builder.add_edge("parse",            "search_swimcloud")
builder.add_edge("search_swimcloud", "reasoning")

builder.set_entry_point("search")
builder.set_finish_point("reasoning")
graph = builder.compile()
#display(Image(graph.get_graph(xray=True).draw_mermaid_png()))


In [100]:
import sys
from pathlib import Path
from sqlalchemy.exc import SQLAlchemyError


sys.path.append(str(Path(os.getcwd()).parent))
from db.db_connection import get_db_session
from db.models import Runner
#print(f"DATABASE_URL: {os.getenv('DATABASE_URL')}")

def get_next_runner():
    session = get_db_session()

    runner = session.query(Runner).filter(Runner.swimmer == None).first()
    first = runner.first_name
    last = runner.last_name
    runner_name = f"{first} {last}"
    college = runner.college_team
    session.close()
    
    return runner_name, college

def save_runner_profile(result: dict) -> None:
    """
    Save runner profile information to the database.
    Uses SQLAlchemy session and upsert logic.
    """
    session = get_db_session()
    try:
        first = result.get("runner_name", "").split()[0].lower()
        last = result.get("runner_name", "").split()[-1].lower()    
        runner = session.query(Runner).filter_by(first_name=first, last_name=last).first()

        if not runner:
            print("Runner not found in database")

        runner.hometown = result.get("hometown")
        runner.high_school = result.get("high_school")
        runner.swim_background = result.get("swim_background")
        runner.runner_url = result.get("selected_profile_url")
        runner.runner_text = result.get("full_profile_text", "")
        runner.swimmer = result.get("has_swim_background", None)
        runner.swim_url = result.get("swim_profile_url")
        runner.swim_text = result.get("swim_profile_text", "")


        session.commit()
        print(f"Saved runner profile for {result.get('runner_name')}")
    except SQLAlchemyError as e:
        print(f"Database error: {e}")
        session.rollback()
    finally:
        session.close()

In [106]:
runner_name, college = get_next_runner()
print(f"Updating runner: {runner_name}, {college}")

Updating runner: sam whitmarsh, Texas A&M


In [107]:
init = RunnerProfileState(runner_name=runner_name, college_name=college)
result = graph.invoke(init)

# Print only the requested fields
print("Runner Name:", result.get("runner_name"))
print("Profile URL:", result.get("selected_profile_url"))
print("Hometown:", result.get("hometown"))
print("High School:", result.get("high_school"))
print("Swimming Experience:", result.get("swim_background"))
print("Swimming Background:", result.get("has_swim_background"))
print("Rationale:", result.get("rationale", "No rationale provided."))

save_runner_profile(result)

Runner Name: sam whitmarsh
Profile URL: https://12thman.com/sports/track/roster/sam-whitmarsh/10530
Hometown: College Station, TX
High School: College Station HS
Swimming Experience: no
Swimming Background: False
Rationale: No profile or swim page text available.
Saved runner profile for sam whitmarsh


In [ ]:
def save_swimcloud_profile(result: dict) -> None:
    """
    Save SwimCloud profile information to the database.
    Stores name, hometown, club names, and profile URL if a match is found.
    Uses SQLAlchemy session and upsert logic.
    """
    session = get_db_session()
    try:
        # Extract swimmer info from result dict
        sc_profile = result.get('swimcloud_profile', {})
        sc_name = sc_profile.get('name', '')
        sc_hometown = sc_profile.get('hometown', '')
        sc_clubs = sc_profile.get('clubs', '')
        sc_url = result.get('swimcloud_url', '')

        # Use runner name for matching, fallback to SwimCloud name if available
        runner_name = result.get('runner_name', sc_name)
        first = runner_name.split()[0].lower()
        last = runner_name.split()[-1].lower()

        runner = session.query(Runner).filter_by(first_name=first, last_name=last).first()

        if not runner:
            print(f"Runner not found in database for SwimCloud save: {runner_name}")
            return

        runner.swimcloud_name = sc_name
        runner.swimcloud_hometown = sc_hometown
        runner.swimcloud_clubs = sc_clubs
        runner.swimcloud_url = sc_url

        session.commit()
        print(f"Saved SwimCloud profile for {runner_name}")
    except SQLAlchemyError as e:
        print(f"Database error: {e}")
        session.rollback()
    finally:
        session.close()

# Usage after running your workflow:
# save_swimcloud_profile(result)

In [ ]:
#Old functions to save for reference
# 1. search_pages
def search_pages(state: RunnerProfileState, config: Any) -> Dict[str, Any]:
    query = f"{state['runner_name']} {state['college_name']} roster profile"
    res = search.invoke(query)
    urls = [hit["url"] for hit in res.get("results", [])]
    snippets = [
        {"url": hit["url"], "snippet": hit.get("content", "")[:2000]}
        for hit in res.get("results", [])
    ]
    return {
        "search_results": urls,
        "page_snippets": snippets
    }